In [1]:
import duckdb
import pandas as pd
import numpy as np

db_path = r"C:\Users\perna\OneDrive\Documents\RussekLab\projects\poker-project1-RL\data\pluribus.duckdb"
con = duckdb.connect(db_path)

print(con.execute("SELECT COUNT(*) FROM preflop").df())

   count_star()
0         61811


In [3]:

# Build clean analysis dataframe
# One row per pre-flop decision
# Only the variables we need for analysis

df = con.execute("""
    SELECT
        true_session,
        hand,
        player_name,
        position,
        hole_cards,
        action,
        amount,
        stack_at_decision,
        finishing_stack,
        blind
    FROM preflop
    ORDER BY true_session, hand, position
""").df()

print(f"Shape: {df.shape}")
print(df.head(10))

Shape: (61811, 10)
  true_session  hand player_name  position hole_cards action  amount  \
0          100     0      MrBlue         0       TcQc     cc     NaN   
1          100     0    MrBlonde         1       8s4c      f     NaN   
2          100     0     MrWhite         2       9c3d      f     NaN   
3          100     0      MrPink         3       Ah4h    cbr   210.0   
4          100     0     MrBrown         4       Th5s      f     NaN   
5          100     0    Pluribus         5       6c7s      f     NaN   
6          100     1    MrBlonde         0       Qh5c      f     NaN   
7          100     1     MrWhite         1       9h6h     cc     NaN   
8          100     1      MrPink         2       KcJh    cbr   210.0   
9          100     1     MrBrown         3       8hQc      f     NaN   

   stack_at_decision  finishing_stack  blind  
0            10000.0          10310.0     50  
1            10000.0           9900.0    100  
2            10000.0          10000.0      0  


In [7]:
# Add last hand context variables
# Shift finishing_stack, action, and blind together to get prior hand info
df['last_hand_profit'] = (
    df.groupby(['true_session', 'player_name'])['finishing_stack'].shift(1) -
    df.groupby(['true_session', 'player_name'])['stack_at_decision'].shift(1)
)
df['last_hand_action'] = df.groupby(['true_session', 'player_name'])['action'].shift(1)
df['last_hand_blind'] = df.groupby(['true_session', 'player_name'])['blind'].shift(1)

# Categorize last hand outcome
def categorize_last_outcome(row):
    profit = row['last_hand_profit']
    action = row['last_hand_action']
    blind = row['last_hand_blind']
    
    if pd.isna(profit):
        return None  # first hand of session
    if profit > 0:
        return 'won'
    if profit < 0:
        return 'lost'
    # profit == 0 cases
    if action == 'f' and blind == 0:
        return 'folded_no_blind'
    if action == 'f' and blind > 0:
        return 'blind_returned'
    if action in ('cc', 'cbr', 'sm'):
        return 'played_broke_even'
    return 'neutral'

df['last_hand_category'] = df.apply(categorize_last_outcome, axis=1)

print(df['last_hand_category'].value_counts(dropna=False))
print(f"\nShape: {df.shape}")

last_hand_category
won                  22096
folded_no_blind      18931
lost                 17922
played_broke_even     2204
NaN                    439
blind_returned         219
Name: count, dtype: int64

Shape: (61811, 15)


In [9]:
# Add binary win/loss for last hand
df['last_hand_outcome'] = np.sign(df['last_hand_profit'])
# -1 = loss, 0 = neutral (folded), 1 = win

print(df['last_hand_outcome'].value_counts())

last_hand_outcome
 1.0    22096
 0.0    21354
-1.0    17922
Name: count, dtype: int64


In [10]:
# Look at the zero profit cases more carefully
zero_profit = df[df['last_hand_profit'] == 0]

print(f"Total zero profit rows: {len(zero_profit)}")
print("\nAction distribution for zero profit last hand:")
print(zero_profit['action'].value_counts())

# What was their action on the PREVIOUS hand that resulted in 0 profit?
# We can infer this from blind - if blind=0 and profit=0, they folded without posting
# if blind=50 or 100 and profit=0, they posted and got it back somehow
print("\nBlind distribution for zero profit last hand:")
print(zero_profit['blind'].value_counts())

Total zero profit rows: 21354

Action distribution for zero profit last hand:
action
f      14910
cbr     3390
cc      2977
sm        77
Name: count, dtype: int64

Blind distribution for zero profit last hand:
blind
0      14126
100     6211
50      1017
Name: count, dtype: int64


In [11]:
print(df.dtypes)
print(f"\nNull counts:")
print(df.isnull().sum())

true_session              str
hand                    int64
player_name               str
position                int64
hole_cards                str
action                    str
amount                float64
stack_at_decision     float64
finishing_stack       float64
blind                   int64
last_hand_profit      float64
last_hand_outcome     float64
last_hand_action          str
last_hand_blind       float64
last_hand_category        str
dtype: object

Null counts:
true_session              0
hand                      0
player_name               0
position                  0
hole_cards                0
action                    0
amount                50965
stack_at_decision         0
finishing_stack           0
blind                     0
last_hand_profit        439
last_hand_outcome       439
last_hand_action        439
last_hand_blind         439
last_hand_category      439
dtype: int64


In [12]:
# Drop redundant column
df = df.drop(columns=['last_hand_outcome'])

# Drop first-hand-of-session rows (no prior history)
df = df.dropna(subset=['last_hand_category'])

print(f"Shape after cleaning: {df.shape}")
print(f"\nNull counts:")
print(df.isnull().sum())

Shape after cleaning: (61372, 14)

Null counts:
true_session              0
hand                      0
player_name               0
position                  0
hole_cards                0
action                    0
amount                50593
stack_at_decision         0
finishing_stack           0
blind                     0
last_hand_profit          0
last_hand_action          0
last_hand_blind           0
last_hand_category        0
dtype: int64


In [13]:
# Save analysis dataframe to DuckDB as preflop_with_history_lag1
con.execute("DROP TABLE IF EXISTS preflop_with_history_lag1")
con.execute("CREATE TABLE preflop_with_history_lag1 AS SELECT * FROM df")
print("saved to duckdb")

# Preview the final table
print("\nTable structure:")
print(con.execute("DESCRIBE preflop_with_history_lag1").df())

print("\nFirst 5 rows:")
print(con.execute("SELECT * FROM preflop_with_history_lag1 LIMIT 5").df())

saved to duckdb

Table structure:
           column_name column_type null   key default extra
0         true_session     VARCHAR  YES  None    None  None
1                 hand      BIGINT  YES  None    None  None
2          player_name     VARCHAR  YES  None    None  None
3             position      BIGINT  YES  None    None  None
4           hole_cards     VARCHAR  YES  None    None  None
5               action     VARCHAR  YES  None    None  None
6               amount      DOUBLE  YES  None    None  None
7    stack_at_decision      DOUBLE  YES  None    None  None
8      finishing_stack      DOUBLE  YES  None    None  None
9                blind      BIGINT  YES  None    None  None
10    last_hand_profit      DOUBLE  YES  None    None  None
11    last_hand_action     VARCHAR  YES  None    None  None
12     last_hand_blind      DOUBLE  YES  None    None  None
13  last_hand_category     VARCHAR  YES  None    None  None

First 5 rows:
  true_session  hand player_name  position hole_car

In [14]:
# Display as a clean readable table
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.1f}'.format)

preview = con.execute("SELECT * FROM preflop_with_history_lag1 LIMIT 10").df()
preview

,true_session,hand,player_name,position,hole_cards,action,amount,stack_at_decision,finishing_stack,blind,last_hand_profit,last_hand_action,last_hand_blind,last_hand_category
0,100,1,MrBlonde,0,Qh5c,f,NaN,9900.0,9950.0,50,-100.0,f,100.0,lost
1,100,1,MrWhite,1,9h6h,cc,NaN,10000.0,9555.0,100,0.0,f,0.0,folded_no_blind
2,100,1,MrPink,2,KcJh,cbr,210.0,9790.0,10495.0,0,-210.0,cbr,0.0,lost
3,100,1,MrBrown,3,8hQc,f,NaN,10000.0,10000.0,0,0.0,f,0.0,folded_no_blind
4,100,1,Pluribus,4,7hKh,f,NaN,10000.0,10000.0,0,0.0,f,0.0,folded_no_blind
5,100,1,MrBlue,5,Ks3c,f,NaN,10310.0,10000.0,0,310.0,cc,50.0,won
6,100,2,MrWhite,0,Jc2c,f,NaN,9555.0,9950.0,50,-445.0,cc,100.0,lost
7,100,2,MrPink,1,2dQh,f,NaN,10495.0,9900.0,100,705.0,cbr,0.0,won
8,100,2,MrBrown,2,9dJh,f,NaN,10000.0,10000.0,0,0.0,f,0.0,folded_no_blind
9,100,2,Pluribus,3,7c8h,f,NaN,10000.0,10000.0,0,0.0,f,0.0,folded_no_blind


In [15]:
import os

output_path = r"C:\Users\perna\OneDrive\Documents\RussekLab\projects\poker-project1-RL\outputs"

# Save preview of final table
preview_full = con.execute("SELECT * FROM preflop_with_history_lag1 LIMIT 20").df()
preview_full.to_csv(os.path.join(output_path, "preflop_with_history_lag1_preview.csv"), index=False)

print("saved to outputs/")

saved to outputs/
